## Tfidf + Logistic Regression Model (baseline model)

In [1]:
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier
from sklearn.metrics import classification_report, f1_score, roc_auc_score

In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer

In [4]:
df= pd.read_csv("C:/01_Data/PythonProgram/Toxic_Comment_Classification_PROJECT/train_data/train_tfidf_clean.csv")

In [3]:
toxic_columns = [
    "toxic",
    "severe_toxic",
    "obscene",
    "threat",
    "insult",
    "identity_hate"
]

In [5]:
X = df["comment_text"]
y = df[toxic_columns]

X_train_text, X_val_text, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    shuffle=True
)

In [6]:
tfidf = TfidfVectorizer(
    max_features=50000,
    ngram_range=(1,2),
    stop_words="english",
    lowercase=True
)

# Fit ONLY on training data
X_train_tfidf = tfidf.fit_transform(X_train_text)

# Transform validation data
X_val_tfidf = tfidf.transform(X_val_text)

In [7]:
log_reg = LogisticRegression(
    solver="liblinear",
    class_weight="balanced",
    max_iter=1000
)

model = OneVsRestClassifier(log_reg)

model.fit(X_train_tfidf, y_train)

OneVsRestClassifier(estimator=LogisticRegression(class_weight='balanced',
                                                 max_iter=1000,
                                                 solver='liblinear'))

In [8]:
y_pred = model.predict(X_val_tfidf)
y_prob = model.predict_proba(X_val_tfidf)

In [9]:
print(classification_report(y_val, y_pred, target_names=toxic_columns))

               precision    recall  f1-score   support

        toxic       0.64      0.85      0.73      3033
 severe_toxic       0.29      0.86      0.43       323
      obscene       0.70      0.87      0.78      1680
       threat       0.22      0.62      0.32        88
       insult       0.55      0.85      0.67      1538
identity_hate       0.24      0.76      0.36       276

    micro avg       0.55      0.85      0.67      6938
    macro avg       0.44      0.80      0.55      6938
 weighted avg       0.59      0.85      0.69      6938
  samples avg       0.06      0.08      0.07      6938



C:\Anaconda\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in samples with no predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Anaconda\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in samples with no true labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Anaconda\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 in samples with no true nor predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [10]:
micro_f1 = f1_score(y_val, y_pred, average="micro")
macro_f1 = f1_score(y_val, y_pred, average="macro")

print("Micro F1:", micro_f1)
print("Macro F1:", macro_f1)

Micro F1: 0.6704545454545454
Macro F1: 0.5473382882968165


In [21]:
threshold = 0.7
y_pred_custom = (y_prob > threshold).astype(int)

print("Micro F1 (0.4 threshold):",
      f1_score(y_val, y_pred_custom, average="micro"))

Micro F1 (0.4 threshold): 0.7139776250168486


In [12]:
roc_auc = roc_auc_score(y_val, y_prob, average="macro")
print("Macro ROC-AUC:", roc_auc)

Macro ROC-AUC: 0.9799708513785212
